In [ ]:
# Установка psycopg2
!pip install psycopg2-binary

# Импорт библиотек
import psycopg2
from psycopg2 import Error
import pandas as pd
from datetime import datetime
import random

print("✅ Все библиотеки установлены и импортированы")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 46.8 MB/s eta 0:00:00


In [ ]:
# Параметры подключения к PostgreSQL
DB_CONFIG = {
    'user': 'postgres',           # Ваш логин
    'password': 'Numberone1!',    # Ваш пароль
    'host': 'localhost',          # Хост
    'port': '5432',               # Порт
    'database': 'postgres'        # База данных
}

print("✅ Конфигурация подключения задана")
print(f"Подключение к: {DB_CONFIG['host']}:{DB_CONFIG['port']}")
print(f"База данных: {DB_CONFIG['database']}")

Соединение установлено! Версия БД: PostgreSQL 16.13 (Ubuntu 16.13-0ubuntu0.24.04.1) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit


In [ ]:
# Задание 1: Подключение к базе данных
connection = None
cursor = None

try:
    connection = psycopg2.connect(**DB_CONFIG)
    cursor = connection.cursor()
    
    print("\n" + "="*50)
    print("ЗАДАНИЕ 1: ПОДКЛЮЧЕНИЕ К БАЗЕ ДАННЫХ")
    print("="*50)
    
    # Получаем версию PostgreSQL
    cursor.execute("SELECT version();")
    version = cursor.fetchone()
    
    print(f"\n✅ Подключение успешно установлено!")
    print(f"Версия PostgreSQL: {version[0]}")
    print(f"Информация о сервере: {connection.get_dsn_parameters()}")
    
except (Exception, Error) as error:
    print(f"❌ Ошибка при подключении: {error}")

finally:
    if connection:
        if cursor:
            cursor.close()
        connection.close()
        print("\nСоединение закрыто")

Схема CRM и таблицы успешно созданы
Соединение закрыто


In [6]:
import psycopg2
from psycopg2 import Error

connection = None
cursor = None

try:
    connection = psycopg2.connect(
        host=DB_HOST,
        user=DB_USER,
        password=DB_PASS,
        dbname=DB_NAME,
        port=DB_PORT
    )

    cursor = connection.cursor()

    cursor.execute("SET search_path TO crm;")

    cursor.execute("""
    INSERT INTO companies (name, industry, website) VALUES
    ('ООО Ромашка', 'Ритейл', 'romashka.com'),
    ('ИП Лучший Сервис', 'Услуги', 'bestservice.ru')
    ON CONFLICT DO NOTHING;
    """)

    cursor.execute("""
    INSERT INTO contacts (company_id, first_name, last_name, email) VALUES
    (1, 'Ольга', 'Иванова', 'o.ivanova@romashka.com'),
    (2, 'Петр', 'Сергеев', 'p.sergeev@bestservice.ru')
    ON CONFLICT DO NOTHING;
    """)

    cursor.execute("""
    INSERT INTO deals (company_id, deal_name, stage, amount) VALUES
    (1, 'Поставка ПО', 'Предложение', 50000),
    (2, 'Консультация', 'Квалификация', 15000)
    ON CONFLICT DO NOTHING;
    """)

    cursor.execute("""
    INSERT INTO activities (contact_id, type, notes) VALUES
    (1, 'Звонок', 'Обсудили требования к ПО'),
    (1, 'Email', 'Отправлено коммерческое предложение'),
    (2, 'Встреча', 'Первичная консультация')
    ON CONFLICT DO NOTHING;
    """)

    connection.commit()
    print("Данные успешно добавлены")

except (Exception, Error) as error:
    print("Ошибка:", error)
    if connection:
        connection.rollback()

finally:
    if connection:
        if cursor:
            cursor.close()
        connection.close()
        print("Соединение закрыто")

Данные успешно добавлены
Соединение закрыто


In [ ]:
# Задание 2: Создание таблиц CRM
connection = None
cursor = None

# SQL код для создания таблиц
create_tables_sql = """
-- Таблица companies (компании)
CREATE TABLE IF NOT EXISTS companies (
    company_id SERIAL PRIMARY KEY,
    name VARCHAR(200) NOT NULL,
    industry VARCHAR(100),
    website VARCHAR(100)
);

-- Таблица contacts (контакты)
CREATE TABLE IF NOT EXISTS contacts (
    contact_id SERIAL PRIMARY KEY,
    company_id INT REFERENCES companies(company_id),
    first_name VARCHAR(50) NOT NULL,
    last_name VARCHAR(50) NOT NULL,
    email VARCHAR(100) UNIQUE
);

-- Таблица deals (сделки)
CREATE TABLE IF NOT EXISTS deals (
    deal_id SERIAL PRIMARY KEY,
    company_id INT REFERENCES companies(company_id),
    deal_name VARCHAR(200) NOT NULL,
    stage VARCHAR(50),
    amount NUMERIC(12,2)
);

-- Таблица activities (активности)
CREATE TABLE IF NOT EXISTS activities (
    activity_id SERIAL PRIMARY KEY,
    contact_id INT REFERENCES contacts(contact_id),
    type VARCHAR(50),
    notes TEXT,
    activity_date DATE DEFAULT CURRENT_DATE
);
"""

try:
    connection = psycopg2.connect(**DB_CONFIG)
    cursor = connection.cursor()
    
    print("\n" + "="*50)
    print("ЗАДАНИЕ 2: СОЗДАНИЕ ТАБЛИЦ")
    print("="*50)
    
    # Создаем таблицы
    cursor.execute(create_tables_sql)
    
    print("\n✅ Таблицы успешно созданы:")
    print("  - companies (компании)")
    print("  - contacts (контакты)")
    print("  - deals (сделки)")
    print("  - activities (активности)")
    
    connection.commit()
    
except (Exception, Error) as error:
    print(f"❌ Ошибка при создании таблиц: {error}")
    if connection:
        connection.rollback()

finally:
    if connection:
        if cursor:
            cursor.close()
        connection.close()
        print("\nСоединение закрыто")

Компании:
(1, 'ООО Ромашка', 'Ритейл', 'romashka.com')
(2, 'ИП Лучший Сервис', 'Услуги', 'bestservice.ru')


In [ ]:
# Задание 3: Заполнение таблиц данными
connection = None
cursor = None

try:
    connection = psycopg2.connect(**DB_CONFIG)
    cursor = connection.cursor()
    
    print("\n" + "="*50)
    print("ЗАДАНИЕ 3: ЗАПОЛНЕНИЕ ТАБЛИЦ ДАННЫМИ")
    print("="*50)
    
    # 1. Заполнение companies (150 компаний)
    print("\n1. Добавление компаний...")
    companies_data = []
    industries = ['IT', 'Ритейл', 'Производство', 'Услуги', 'Финансы', 'Страхование', 
                  'Логистика', 'Медицина', 'Образование', 'Строительство']
    
    for i in range(1, 151):
        name = f"Компания_{i}"
        industry = random.choice(industries)
        website = f"www.company{i}.ru"
        companies_data.append((name, industry, website))
    
    cursor.executemany("""
        INSERT INTO companies (name, industry, website) 
        VALUES (%s, %s, %s)
    """, companies_data)
    print(f"   ✅ Добавлено {len(companies_data)} компаний")
    
    # 2. Заполнение contacts (от 2 до 5 контактов на компанию)
    print("\n2. Добавление контактов...")
    first_names = ['Иван', 'Анна', 'Петр', 'Елена', 'Сергей', 'Ольга', 'Дмитрий', 'Мария', 'Алексей', 'Татьяна']
    last_names = ['Иванов', 'Петрова', 'Сидоров', 'Козлова', 'Смирнов', 'Новикова', 'Морозов', 'Волкова']
    
    contacts_data = []
    for company_id in range(1, 151):
        num_contacts = random.randint(2, 5)
        for j in range(num_contacts):
            first_name = random.choice(first_names)
            last_name = random.choice(last_names)
            email = f"{first_name.lower()}.{last_name.lower()}{j}@company{company_id}.ru"
            contacts_data.append((company_id, first_name, last_name, email))
    
    cursor.executemany("""
        INSERT INTO contacts (company_id, first_name, last_name, email) 
        VALUES (%s, %s, %s, %s)
    """, contacts_data)
    print(f"   ✅ Добавлено {len(contacts_data)} контактов")
    
    # 3. Заполнение deals (от 1 до 4 сделок на компанию)
    print("\n3. Добавление сделок...")
    stages = ['Квалификация', 'Предложение', 'Переговоры', 'Закрытие выигрыш', 'Закрытие проигрыш']
    
    deals_data = []
    for company_id in range(1, 151):
        num_deals = random.randint(1, 4)
        for j in range(num_deals):
            deal_name = f"Сделка_{company_id}_{j+1}"
            stage = random.choice(stages)
            amount = random.randint(10000, 5000000)
            deals_data.append((company_id, deal_name, stage, amount))
    
    cursor.executemany("""
        INSERT INTO deals (company_id, deal_name, stage, amount) 
        VALUES (%s, %s, %s, %s)
    """, deals_data)
    print(f"   ✅ Добавлено {len(deals_data)} сделок")
    
    # 4. Заполнение activities (от 5 до 15 активностей на контакт)
    print("\n4. Добавление активностей...")
    activity_types = ['Звонок', 'Email', 'Встреча', 'Демонстрация', 'Презентация', 'Отправка КП']
    notes_templates = ['Обсудили условия', 'Отправили КП', 'Провели встречу', 'Показали продукт', 'Согласовали детали']
    
    # Получаем все contact_id
    cursor.execute("SELECT contact_id FROM contacts")
    contact_ids = [row[0] for row in cursor.fetchall()]
    
    activities_data = []
    for contact_id in contact_ids:
        num_activities = random.randint(5, 15)
        for _ in range(num_activities):
            activity_type = random.choice(activity_types)
            notes = random.choice(notes_templates)
            activity_date = f"2024-{random.randint(1, 12)}-{random.randint(1, 28)}"
            activities_data.append((contact_id, activity_type, notes, activity_date))
    
    cursor.executemany("""
        INSERT INTO activities (contact_id, type, notes, activity_date) 
        VALUES (%s, %s, %s, %s)
    """, activities_data)
    print(f"   ✅ Добавлено {len(activities_data)} активностей")
    
    connection.commit()
    
    # Статистика
    print("\n" + "="*50)
    print("СТАТИСТИКА ЗАПОЛНЕНИЯ:")
    print("="*50)
    
    cursor.execute("SELECT COUNT(*) FROM companies")
    print(f"Компаний: {cursor.fetchone()[0]}")
    
    cursor.execute("SELECT COUNT(*) FROM contacts")
    print(f"Контактов: {cursor.fetchone()[0]}")
    
    cursor.execute("SELECT COUNT(*) FROM deals")
    print(f"Сделок: {cursor.fetchone()[0]}")
    
    cursor.execute("SELECT COUNT(*) FROM activities")
    print(f"Активностей: {cursor.fetchone()[0]}")
    
    print("\n✅ Задание 3 выполнено: все таблицы заполнены данными (100+ записей)")
    
except (Exception, Error) as error:
    print(f"❌ Ошибка при добавлении данных: {error}")
    if connection:
        connection.rollback()

finally:
    if connection:
        if cursor:
            cursor.close()
        connection.close()
        print("\nСоединение закрыто")

Сделки с компаниями:
('Поставка ПО', 'Предложение', Decimal('50000'), 'ООО Ромашка')
('Консультация', 'Квалификация', Decimal('15000'), 'ИП Лучший Сервис')


In [ ]:
# Задание 4: Простой SELECT запрос
connection = None
cursor = None

try:
    connection = psycopg2.connect(**DB_CONFIG)
    cursor = connection.cursor()
    
    print("\n" + "="*50)
    print("ЗАДАНИЕ 4: ПРОСТОЙ SELECT ЗАПРОС")
    print("="*50)
    
    # Вывод первых 10 компаний
    print("\n📊 Первые 10 компаний:")
    print("-"*80)
    cursor.execute("SELECT company_id, name, industry, website FROM companies LIMIT 10;")
    records = cursor.fetchall()
    
    print(f"{'ID':<5} {'Название компании':<25} {'Отрасль':<15} {'Сайт':<25}")
    print("-"*80)
    for row in records:
        print(f"{row[0]:<5} {row[1]:<25} {row[2]:<15} {row[3]:<25}")
    
    # Статистика по таблицам
    print("\n📊 Статистика по таблицам:")
    print("-"*50)
    cursor.execute("""
        SELECT 
            (SELECT COUNT(*) FROM companies) as companies,
            (SELECT COUNT(*) FROM contacts) as contacts,
            (SELECT COUNT(*) FROM deals) as deals,
            (SELECT COUNT(*) FROM activities) as activities
    """)
    stats = cursor.fetchone()
    print(f"Компаний: {stats[0]}")
    print(f"Контактов: {stats[1]}")
    print(f"Сделок: {stats[2]}")
    print(f"Активностей: {stats[3]}")
    
    print("\n✅ Задание 4 выполнено")
    
except (Exception, Error) as error:
    print(f"❌ Ошибка: {error}")

finally:
    if connection:
        if cursor:
            cursor.close()
        connection.close()
        print("\nСоединение закрыто")

In [ ]:
# Задание 5: JOIN запрос и экспорт в CSV
connection = None
cursor = None

try:
    connection = psycopg2.connect(**DB_CONFIG)
    cursor = connection.cursor()
    
    print("\n" + "="*50)
    print("ЗАДАНИЕ 5: JOIN ЗАПРОС И ЭКСПОРТ В CSV")
    print("="*50)
    
    # JOIN запрос для аналитики
    join_query = """
    SELECT 
        c.name AS компания,
        c.industry AS отрасль,
        COUNT(DISTINCT cont.contact_id) AS количество_контактов,
        COUNT(DISTINCT d.deal_id) AS количество_сделок,
        COUNT(DISTINCT CASE WHEN d.stage = 'Закрытие выигрыш' THEN d.deal_id END) AS выигранные_сделки,
        COUNT(DISTINCT CASE WHEN d.stage = 'Закрытие проигрыш' THEN d.deal_id END) AS проигранные_сделки,
        ROUND(COUNT(DISTINCT CASE WHEN d.stage = 'Закрытие выигрыш' THEN d.deal_id END) * 100.0 / 
              NULLIF(COUNT(DISTINCT d.deal_id), 0), 2) AS процент_выигрыша,
        COUNT(DISTINCT a.activity_id) AS количество_активностей
    FROM companies c
    LEFT JOIN contacts cont ON c.company_id = cont.company_id
    LEFT JOIN deals d ON c.company_id = d.company_id
    LEFT JOIN activities a ON cont.contact_id = a.contact_id
    GROUP BY c.name, c.industry
    ORDER BY выигранные_сделки DESC, количество_сделок DESC
    """
    
    print("\n📊 Выполнение JOIN запроса...")
    cursor.execute(join_query)
    records = cursor.fetchall()
    colnames = [desc[0] for desc in cursor.description]
    
    # Создаем DataFrame
    df = pd.DataFrame(records, columns=colnames)
    
    # Выводим первые 20 результатов
    print("\n📊 Первые 20 компаний по эффективности:")
    print("="*120)
    print(df.head(20).to_string(index=False))
    
    # Сохраняем в CSV
    csv_filename = "crm_report_for_datalens.csv"
    df.to_csv(csv_filename, index=False, encoding='utf-8-sig')
    
    print("\n" + "="*50)
    print("💾 ФАЙЛ СОХРАНЕН!")
    print("="*50)
    print(f"Имя файла: {csv_filename}")
    print(f"Количество записей: {len(df)}")
    print(f"Количество полей: {len(df.columns)}")
    print(f"Поля: {', '.join(df.columns)}")
    
    # Скачивание файла
    from google.colab import files
    files.download(csv_filename)
    
    print("\n✅ Задание 5 выполнено: CSV файл создан и скачан")
    
except (Exception, Error) as error:
    print(f"❌ Ошибка: {error}")

finally:
    if connection:
        if cursor:
            cursor.close()
        connection.close()
        print("\nСоединение закрыто")